# MoDANet trên Google Colab — notebook all-in-one

Doan et al., *"MoDANet: Multi-Task Deep Network for Joint Automatic Modulation Classification and Direction of Arrival Estimation"*, IEEE Comm. Letters 26(2), 2022.

Notebook này gom **toàn bộ code** của `modanet/` (configs, model, dữ liệu, train, đánh giá, suy luận) vào một file để chạy trên Colab, không cần upload thư mục code. Kiến trúc, siêu tham số và định dạng checkpoint giống hệt bản local (theo `MODANET_INSTRUCTION.md`):
Type A, K = 64, filter 7×1, conv output 1×1, **457,413 tham số**, SGD momentum 0.9, lr 0.01 (×0.1 sau epoch 30), batch hiệu dụng 128, 60 epoch, early stopping patience 8.

**Checkpoint dùng chung được hai chiều:** `best.pt` / `last.pt` train trên Colab chạy được với `scripts/inference.py` và `scripts/train.py --resume` ở máy local, và ngược lại (upload `last.pt` từ máy local lên Drive thì notebook train tiếp).

## Các bước

| # | Việc | Ghi chú |
|---|---|---|
| 0 | Chọn GPU: **Runtime → Change runtime type → T4 GPU** | bắt buộc làm trước khi chạy cell nào |
| 1 | Sửa ô **CẤU HÌNH** | thường chỉ cần kiểm tra `DRIVE_DIR` |
| 2 | **Runtime → Run all** (Ctrl+F9) | mount Drive → chép dữ liệu → kiểm model → smoke test → train đầy đủ → đánh giá |
| 3 | Bị ngắt kết nối? Chạy lại **Run all** | tự resume từ `last.pt` trên Drive |

Cần chuẩn bị trước trên Google Drive (xem `HUONG_DAN_COLAB.md` cùng thư mục):
```
MyDrive/modanet/data_processed/x_float16.npy        (9.22 GB — hoặc các phần x_float16.npy.part_aa, _ab, ... tạo bằng `split -b 1G`)
                              labels.npy, meta.json, SHA256SUMS
                              splits/random_seed42.npz
```
Kết quả được ghi vào `MyDrive/modanet/outputs/<RUN_NAME>/` (log.csv, last.pt, best.pt, hình, kết quả test), nên không mất khi Colab ngắt.

## 0. Kiểm tra GPU
Nếu dòng dưới báo `NVIDIA-SMI has failed` hoặc không có GPU: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**, rồi chạy lại.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch, numpy
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| numpy", numpy.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")

## 1. CẤU HÌNH — chỉ cần sửa ô này

| Biến | Ý nghĩa |
|---|---|
| `DRIVE_DIR` | thư mục trên Drive chứa `data_processed/` (và nơi ghi `outputs/`) |
| `DATA_SOURCE` | `"processed"`: dùng memmap đã convert (khuyên dùng). `"mat_zip"`: convert từ file zip `.mat` gốc trên Drive (mục 5B, cần Drive ≥ 30 GB) |
| `RUN_SMOKE` | chạy thử 1 epoch trên 20k mẫu trước khi train thật (~vài phút) để đo tốc độ |
| `RUN_FULL` | train đầy đủ 60 epoch; tự resume nếu `outputs/<RUN_NAME>/last.pt` đã có |
| `MAX_HOURS` | dừng an toàn trước khi hết phiên Colab (free ~12 giờ, thường ngắt sớm hơn). Chạy lại để train tiếp |
| `IN_MEMORY` | `"auto"`: nạp 9.2 GB vào RAM chỉ khi dư ≥ 3 GB (Colab Pro High-RAM). Colab free (12.7 GB RAM) → đọc memmap từ đĩa `/content` |
| `SHUFFLE` | `"chunk"` (giống local, mặc định) hoặc `"random"` (hoán vị thuần, nên dùng khi `IN_MEMORY` bật) |

In [ ]:
import os, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# ---- Đường dẫn ----
DRIVE_DIR   = "/content/drive/MyDrive/modanet"        # thư mục trên Google Drive
DATA_SOURCE = "processed"                             # "processed" | "mat_zip"
MAT_ZIP     = f"{DRIVE_DIR}/MoDANet-dataset.zip"      # chỉ dùng khi DATA_SOURCE = "mat_zip"
LOCAL_DATA  = "/content/data_processed"               # bản sao trên đĩa của máy Colab (nhanh hơn đọc thẳng từ Drive)
OUT_ROOT    = f"{DRIVE_DIR}/outputs"                  # checkpoint + log ghi thẳng lên Drive
RUN_NAME    = "run01"

# ---- Việc cần chạy ----
RUN_SMOKE = True        # chạy thử nhanh trước (đo thời gian/epoch)
RUN_FULL  = True        # train đầy đủ (tự resume)
RUN_EVAL  = True        # đánh giá best.pt trên tập test sau khi train

# ---- Train (mặc định = cấu hình bài báo / instruction; chỉ đổi nếu biết mình đang làm gì) ----
PRESET        = "paper"      # paper | medium | small | tiny
SPLIT_MODE    = "random"     # random (như bài) | group (khắt khe hơn)
EPOCHS        = 60
BATCH_SIZE    = "auto"       # micro-batch: "auto" theo VRAM (T4 16 GB -> 128×1), batch hiệu dụng luôn 128
NO_EARLY_STOP = False        # True = train đủ 60 epoch như MCNet
MAX_HOURS     = 11.0         # None = không giới hạn
IN_MEMORY     = "auto"       # "auto" | True | False
SHUFFLE       = "chunk"      # "chunk" | "random"
NUM_WORKERS   = "auto"
LOG_EVERY     = 500          # in tiến độ mỗi N iteration (Colab chậm nếu in quá nhiều)
VERIFY_SHA256 = True         # kiểm checksum 1 lần sau khi chép (~1–2 phút), bắt file upload bị hỏng

if not IN_COLAB:   # chạy ngoài Colab (máy local / Jupyter): sửa đường dẫn cho phù hợp
    VERIFY_SHA256 = False
    DRIVE_DIR  = os.environ.get("MODANET_DRIVE_DIR", os.path.abspath(".."))
    LOCAL_DATA = os.environ.get("MODANET_PROCESSED_DIR", os.path.join(DRIVE_DIR, "data_processed"))
    OUT_ROOT   = os.path.join(DRIVE_DIR, "outputs")
print("IN_COLAB =", IN_COLAB, "| data ->", LOCAL_DATA, "| outputs ->", OUT_ROOT)

## 2. Mount Google Drive và chép dữ liệu sang đĩa của máy Colab
Colab sẽ hỏi quyền truy cập Drive: chọn tài khoản có thư mục `modanet/` rồi **Allow**.

Đọc ngẫu nhiên memmap 9.2 GB trực tiếp từ Drive (qua mạng) rất chậm, nên notebook chép một lần sang `/content/data_processed` (~3–10 phút), ghép lại nếu upload thành nhiều phần, rồi kiểm SHA256. Nếu chỉ restart runtime (không ngắt máy) thì bản sao vẫn còn và bước này được bỏ qua.

In [ ]:
import shutil, time, hashlib, json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

SRC_PROCESSED = f"{DRIVE_DIR}/data_processed"
DATA_FILES = ["meta.json", "labels.npy", "SHA256SUMS", "splits/random_seed42.npz", "splits/group_seed42.npz",
              "x_float16.npy"]


def copy_with_progress(src, dst, chunk=64 << 20):
    """Chép file lớn, có thanh tiến độ; bỏ qua nếu đích đã đủ kích thước (chép dở thì chép lại)."""
    size = os.path.getsize(src)
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    from tqdm.auto import tqdm
    tmp = dst + ".part"
    with open(src, "rb") as fi, open(tmp, "wb") as fo, \
            tqdm(total=size, unit="B", unit_scale=True, desc=os.path.basename(src)) as bar:
        while True:
            b = fi.read(chunk)
            if not b:
                break
            fo.write(b)
            bar.update(len(b))
    os.replace(tmp, dst)
    return True


def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def join_parts(parts, dst, chunk=64 << 20):
    """Ghép x_float16.npy.part_aa, _ab, ... (tạo bằng `split -b 1G`) thành 1 file."""
    total = sum(os.path.getsize(p) for p in parts)
    if os.path.exists(dst) and os.path.getsize(dst) == total:
        return False
    from tqdm.auto import tqdm
    tmp = dst + ".part"
    with open(tmp, "wb") as fo, tqdm(total=total, unit="B", unit_scale=True, desc=f"ghép {len(parts)} phần") as bar:
        for p in parts:
            with open(p, "rb") as fi:
                for b in iter(lambda: fi.read(chunk), b""):
                    fo.write(b)
                    bar.update(len(b))
    os.replace(tmp, dst)
    return True


def fetch_processed(src_dir, dst_dir, verify=True):
    import glob
    big = os.path.join(src_dir, "x_float16.npy")
    parts = sorted(glob.glob(big + ".part_*"))
    if not os.path.exists(big) and not parts:
        raise FileNotFoundError(
            f"Không thấy {big} (hay các phần {os.path.basename(big)}.part_*). Upload thư mục data_processed/ "
            f"lên Drive (xem HUONG_DAN_COLAB.md), hoặc đặt DATA_SOURCE = 'mat_zip'.")
    t = time.time()
    for f in DATA_FILES:
        s = os.path.join(src_dir, f)
        if os.path.exists(s) and os.path.abspath(s) != os.path.abspath(os.path.join(dst_dir, f)):
            copy_with_progress(s, os.path.join(dst_dir, f))
    if not os.path.exists(big) and parts:
        join_parts(parts, os.path.join(dst_dir, "x_float16.npy"))
    print(f"Dữ liệu sẵn sàng ở {dst_dir} ({time.time() - t:.0f}s)")
    sums, mark = os.path.join(dst_dir, "SHA256SUMS"), os.path.join(dst_dir, ".sha256_ok")
    if verify and os.path.exists(sums) and not os.path.exists(mark):
        for line in open(sums):
            want, name = line.split()
            got = sha256_file(os.path.join(dst_dir, name))
            print(f"  {'OK ' if got == want else 'SAI'} {name}")
            if got != want:
                raise RuntimeError(f"{name} hỏng khi upload/chép — xoá bản trên Drive và upload lại")
        open(mark, "w").close()   # đã kiểm -> lần sau (cùng máy Colab) bỏ qua


if DATA_SOURCE == "processed":
    fetch_processed(SRC_PROCESSED, LOCAL_DATA, VERIFY_SHA256)
    print(json.load(open(os.path.join(LOCAL_DATA, "meta.json")))["shape"])
!df -h /content | tail -1

## 3. Code: hằng số (configs/)
Giống `configs/model_config.py`, `configs/data_config.py`, `configs/train_config.py`. Nhãn xuất xứ: [TABLE-I] [TEXT] [FIG] [DERIVED] [GAP].

In [ ]:
# ================= configs/model_config.py =================
M_ANTENNAS   = 5      # [TEXT] ULA 5 phần tử
N_SAMPLES    = 1024   # [TEXT] số mẫu envelope phức
IQ_CHANNELS  = 2      # [TEXT] I và Q
K_FILTERS    = 64     # [TEXT] "K = 64 filters in all conv layers"
FILTER_SIZE  = 7      # [TABLE-I] ghi 3, [TEXT] chốt 7×1
FILTER_SIZE_CHOICES = (3, 5, 7, 9)
SKIP_CONV_SIZE   = 1  # [TABLE-I] 1×1, không đổi theo FILTER_SIZE
OUTPUT_CONV_SIZE = 1  # [TABLE-I] instruction 1.3: luôn 1×1
KERNEL_WIDTH = 1      # mọi kernel k×1 -> không trượt qua trục I/Q
CONV_STRIDE  = 1
POOL_BACKBONE_SIZE, POOL_BACKBONE_STRIDE, POOL_BACKBONE_PAD = 3, 2, 1   # [TABLE-I] -> ceil(L/2)
POOL_SKIP_SIZE, POOL_SKIP_STRIDE, POOL_SKIP_PAD = 5, 4, 2               # [TABLE-I] -> ceil(L/4)
POOL_STRIDE_W = 1
PAD_W = 0
BACKBONE_GROUPS = 2   # [TEXT] "two four-layer groups"
SKIP_GROUPS     = 1
VARIANT = "A"
VARIANT_SHARED_BLOCKS = {"A": 1, "B": 2, "C": 3}   # [FIG 3a]
TOTAL_RESBLOCK_DEPTH  = 3
NUM_MOD_CLASSES = 12
NUM_DOA_CLASSES = 121
FLATTEN_START_DIM = 1
LOSS_WEIGHT_AMC = 1.0
LOSS_WEIGHT_DOA = 1.0
PRESETS = {
    "paper":  {"K": 64, "k": 7, "variant": "A"},
    "medium": {"K": 64, "k": 5, "variant": "A"},
    "small":  {"K": 32, "k": 5, "variant": "A"},
    "tiny":   {"K": 16, "k": 3, "variant": "A"},
}
DEFAULT_PRESET = "paper"
EXPECTED_PARAM_COUNT = {("A", 3): 292_293, ("A", 5): 374_853, ("A", 7): 457_413, ("A", 9): 539_973,
                        ("B", 7): 395_397, ("C", 7): 333_381}
EXPECTED_PARAM_COUNT_PRESET = {"paper": 457_413, "medium": 374_853, "small": 129_125, "tiny": 44_757}


def same_pad(k):
    return (k - 1) // 2


def pool_out_len(L, size, stride, pad):
    return (L + 2 * pad - size) // stride + 1


# ================= configs/data_config.py =================
# ⚠️ Thứ tự theo CHỈ SỐ TÊN FILE (đã kiểm trên dataset đầy đủ), KHÁC thứ tự bài báo liệt kê.
MODULATIONS = ["16QAM", "64QAM", "PSK", "QPSK", "8PSK", "QFSK",
               "8FSK", "16APSK", "4PAM", "LFM", "DSB-SC", "SSB-SC"]
DOA_MIN, DOA_MAX, DOA_STEP = -60, 60, 1
DOA_ANGLES = list(range(DOA_MIN, DOA_MAX + DOA_STEP, DOA_STEP))   # 121 góc
SNR_MIN, SNR_MAX, SNR_STEP = -10, 20, 1
SNR_LEVELS = list(range(SNR_MIN, SNR_MAX + SNR_STEP, SNR_STEP))   # 31 mức
N_MOD, N_DOA, N_SNR, N_REP = len(MODULATIONS), len(DOA_ANGLES), len(SNR_LEVELS), 10
N_TOTAL = N_MOD * N_DOA * N_SNR * N_REP   # 450,120
MAT_VARIABLE = "data"
RAW_SHAPE = (1024, 5, 2)          # (N, M, IQ) trong file .mat
PERMUTE_RAW_TO_TORCH = (1, 0, 2)  # -> (M, N, IQ)
SAMPLE_SHAPE = (5, 1024, 2)
FILENAME_REGEX = r"^data(\d+)\.mat$"
LABEL_COLUMNS = ("mod_idx", "doa_idx", "snr_db")
COL_MOD, COL_DOA, COL_SNR = 0, 1, 2
NORMALIZE = False                 # [GAP] bài không chuẩn hóa
TEST_FRACTION = 0.2               # [TEXT] 80/20
VAL_FRACTION_OF_TRAIN = 0.1       # [GAP] val tách từ train
SPLIT_SEED = 42
SPLIT_MODES = ("random", "group")
CONVERT_BATCH = 2000


def idx_to_angle(doa_idx):
    return DOA_ANGLES[0] + doa_idx * DOA_STEP


# ================= configs/train_config.py =================  (toàn bộ [GAP], theo MCNet)
LR, MOMENTUM = 0.01, 0.9
LR_DROP_EPOCH, LR_DROP_FACTOR = 30, 0.1
TRAIN_EPOCHS = 60
EFFECTIVE_BATCH = 128
SEED = 42
PATIENCE = 8
AUTO_BATCH_BY_VRAM_GB = ((6.0, 128), (3.5, 64), (1.5, 32), (0.0, 16))
MAX_WORKERS = 4
PIN_MEMORY, PREFETCH_FACTOR = True, 4
SHUFFLE_MODES = ("chunk", "random")
CHUNK_SIZE, SHUFFLE_GROUP_CHUNKS = 64, 64
SMOKE_SUBSET, SMOKE_EPOCHS = 2000, 2
OVERFIT_WARN_EPOCHS = 5

## 4. Code: model (src/models/)
Layout tensor `(B, C, N, IQ)`. Y-shape Type A:
```
input -> InputBlock -> RB1 (chung) -+-> RB2a -> RB3a -> OutputBlock -> 12 logits  (AMC)
                                    +-> RB2b -> RB3b -> OutputBlock -> 121 logits (DOA)
```

In [ ]:
import torch
import torch.nn as nn


def make_conv(cin, cout, k):
    """conv k×1, stride 1, 'same' padding, có bias."""
    return nn.Conv2d(cin, cout, kernel_size=(k, KERNEL_WIDTH), stride=(CONV_STRIDE, CONV_STRIDE),
                     padding=(same_pad(k), PAD_W), bias=True)


def _in_channels(cin, K, n_groups):
    return ([cin] + [K] * n_groups)[:n_groups]


class ConvUnit(nn.Module):
    """conv -> pool -> norm -> relu   [TEXT] Fig.2b (pool ĐỨNG TRƯỚC norm)."""

    def __init__(self, cin, cout, k, pool_k, pool_s, pool_p):
        super().__init__()
        self.conv = make_conv(cin, cout, k)
        self.pool = nn.MaxPool2d(kernel_size=(pool_k, KERNEL_WIDTH), stride=(pool_s, POOL_STRIDE_W),
                                 padding=(pool_p, PAD_W))
        self.norm = nn.BatchNorm2d(cout)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.norm(self.pool(self.conv(x))))


class InputBlock(ConvUnit):
    def __init__(self, cin, K, k):
        super().__init__(cin, K, k, POOL_BACKBONE_SIZE, POOL_BACKBONE_STRIDE, POOL_BACKBONE_PAD)


class ResBlock(nn.Module):
    """backbone (2 nhóm, conv k×1) + skip (1 nhóm, conv 1×1, pool 5×1 s4) -> add."""

    def __init__(self, cin, K, k):
        super().__init__()
        self.backbone = nn.Sequential(*[
            ConvUnit(c, K, k, POOL_BACKBONE_SIZE, POOL_BACKBONE_STRIDE, POOL_BACKBONE_PAD)
            for c in _in_channels(cin, K, BACKBONE_GROUPS)])
        self.skip = nn.Sequential(*[
            ConvUnit(c, K, SKIP_CONV_SIZE, POOL_SKIP_SIZE, POOL_SKIP_STRIDE, POOL_SKIP_PAD)
            for c in _in_channels(cin, K, SKIP_GROUPS)])

    def forward(self, x):
        b, s = self.backbone(x), self.skip(x)
        if b.shape != s.shape:
            raise RuntimeError(f"backbone {tuple(b.shape)} != skip {tuple(s.shape)}")
        return b + s


class OutputBlock(nn.Module):
    """conv -> norm -> relu -> fc. Trả LOGITS (softmax chỉ áp khi suy luận)."""

    def __init__(self, K, flat_dim, n_classes, conv_k=OUTPUT_CONV_SIZE):
        super().__init__()
        self.conv = make_conv(K, K, conv_k)
        self.norm = nn.BatchNorm2d(K)
        self.act = nn.ReLU()
        self.fc = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        return self.fc(torch.flatten(self.act(self.norm(self.conv(x))), start_dim=FLATTEN_START_DIM))


def final_length(N):
    L = pool_out_len(N, POOL_BACKBONE_SIZE, POOL_BACKBONE_STRIDE, POOL_BACKBONE_PAD)
    for _ in range(TOTAL_RESBLOCK_DEPTH):
        L = pool_out_len(L, POOL_SKIP_SIZE, POOL_SKIP_STRIDE, POOL_SKIP_PAD)
    return L


class MoDANet(nn.Module):
    def __init__(self, M=M_ANTENNAS, K=K_FILTERS, k=FILTER_SIZE, N=N_SAMPLES,
                 n_amc=NUM_MOD_CLASSES, n_doa=NUM_DOA_CLASSES, variant=VARIANT,
                 output_conv_k=OUTPUT_CONV_SIZE):
        super().__init__()
        if k not in FILTER_SIZE_CHOICES:
            raise ValueError(f"filter size {k} không thuộc {FILTER_SIZE_CHOICES}")
        if variant not in VARIANT_SHARED_BLOCKS:
            raise ValueError(f"variant {variant!r} không thuộc {tuple(VARIANT_SHARED_BLOCKS)}")
        self.hparams = dict(M=M, K=K, k=k, N=N, n_amc=n_amc, n_doa=n_doa,
                            variant=variant, output_conv_k=output_conv_k)
        self.n_shared = VARIANT_SHARED_BLOCKS[variant]
        self.n_branch = TOTAL_RESBLOCK_DEPTH - self.n_shared
        self.input_block = InputBlock(M, K, k)
        self.shared = nn.Sequential(*[ResBlock(K, K, k) for _ in range(self.n_shared)])
        self.amc_br = nn.Sequential(*[ResBlock(K, K, k) for _ in range(self.n_branch)])
        self.doa_br = nn.Sequential(*[ResBlock(K, K, k) for _ in range(self.n_branch)])
        self.final_len = final_length(N)
        self.flat_dim = K * self.final_len * IQ_CHANNELS
        self.amc_head = OutputBlock(K, self.flat_dim, n_amc, output_conv_k)
        self.doa_head = OutputBlock(K, self.flat_dim, n_doa, output_conv_k)

    def forward(self, x):                    # x: (B, M, N, IQ)
        h = self.shared(self.input_block(x))
        return self.amc_head(self.amc_br(h)), self.doa_head(self.doa_br(h))


class MultiTaskLoss(nn.Module):
    """L = L_AMC + L_DOA (cross-entropy mỗi nhánh)."""

    def __init__(self, w_amc=LOSS_WEIGHT_AMC, w_doa=LOSS_WEIGHT_DOA):
        super().__init__()
        self.w_amc, self.w_doa = w_amc, w_doa
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits_amc, logits_doa, y_amc, y_doa):
        l_amc, l_doa = self.ce(logits_amc, y_amc), self.ce(logits_doa, y_doa)
        return self.w_amc * l_amc + self.w_doa * l_doa, l_amc, l_doa


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ---- kiểm nhanh: số tham số + shape khớp giá trị vàng của instruction ----
for (v, k), n in EXPECTED_PARAM_COUNT.items():
    assert count_params(MoDANet(k=k, variant=v)) == n, (v, k)
for name, n in EXPECTED_PARAM_COUNT_PRESET.items():
    p = PRESETS[name]
    assert count_params(MoDANet(K=p["K"], k=p["k"], variant=p["variant"])) == n, name
_m = MoDANet().eval()
with torch.no_grad():
    _a, _d = _m(torch.randn(2, *SAMPLE_SHAPE))
assert _a.shape == (2, NUM_MOD_CLASSES) and _d.shape == (2, NUM_DOA_CLASSES)
print(f"Model OK: {count_params(_m):,} tham số (Type A, K=64, k=7) | logits {tuple(_a.shape)}, {tuple(_d.shape)}")
del _m, _a, _d

## 5. Code: dữ liệu (src/data/)
Nhãn giải mã từ chỉ số tên file: `index = ((snr_i·12 + mod_i)·121 + doa_i)·10 + rep` (1-based). Memmap hàng `r` ↔ `data{r+1:08d}.mat`.

In [ ]:
import re
import numpy as np
from torch.utils.data import Dataset, Sampler

_FILE_RE = re.compile(FILENAME_REGEX)


def decode_index(index1based):
    """chỉ số tên file -> (snr_dB, mod_idx, doa_idx, rep). Chạy được với mảng numpy."""
    z = index1based - 1
    rep = z % N_REP
    c = z // N_REP
    doa_i = c % N_DOA
    c = c // N_DOA
    return c // N_MOD + SNR_MIN, c % N_MOD, doa_i, rep + 1


def index_from_filename(name):
    m = _FILE_RE.match(os.path.basename(name))
    return int(m.group(1)) if m else None


def labels_for_all_rows():
    snr, mod, doa, _ = decode_index(np.arange(1, N_TOTAL + 1, dtype=np.int64))
    out = np.empty((N_TOTAL, len(LABEL_COLUMNS)), dtype=np.int16)
    out[:, COL_MOD], out[:, COL_DOA], out[:, COL_SNR] = mod, doa, snr
    return out


# ---------------- memmap ----------------
def open_processed(processed_dir, dtype=None):
    meta_p = os.path.join(processed_dir, "meta.json")
    if not os.path.exists(meta_p):
        raise FileNotFoundError(f"Chưa có dữ liệu đã convert trong {processed_dir} (chạy mục 2 / 5B)")
    with open(meta_p) as f:
        meta = json.load(f)
    if not meta.get("complete"):
        raise RuntimeError("convert chưa xong")
    if meta["modulations"] != MODULATIONS:
        raise RuntimeError("thứ tự MODULATIONS trong meta.json khác cấu hình")
    dtype = dtype or meta["dtype"]
    x = np.load(os.path.join(processed_dir, f"x_{dtype}.npy"), mmap_mode="r")
    labels = np.load(os.path.join(processed_dir, "labels.npy"))
    return x, labels, meta


class ModaNetDataset(Dataset):
    """ds[mảng vị trí] -> nguyên 1 batch bằng MỘT lần fancy-index memmap (vị trí đã sắp tăng)."""

    def __init__(self, x, labels, rows):
        self.x = x
        self.rows = np.asarray(rows, dtype=np.int64)
        self.y_mod = torch.from_numpy(labels[self.rows, COL_MOD].astype(np.int64))
        self.y_doa = torch.from_numpy(labels[self.rows, COL_DOA].astype(np.int64))
        self.snr = torch.from_numpy(labels[self.rows, COL_SNR].astype(np.int64))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, pos):
        if np.isscalar(pos):
            xb = torch.from_numpy(np.asarray(self.x[self.rows[pos]], dtype=np.float32))
            return xb, self.y_mod[pos], self.y_doa[pos], self.snr[pos]
        pos = np.sort(np.asarray(pos, dtype=np.int64))
        xb = torch.from_numpy(np.asarray(self.x[self.rows[pos]], dtype=np.float32))
        t = torch.from_numpy(pos)
        return xb, self.y_mod[t], self.y_doa[t], self.snr[t]


class ChunkShuffleBatchSampler(Sampler):
    """chunk_size=1 -> hoán vị thuần. Ngược lại: xáo khối liền kề, gom nhóm, xáo trong nhóm."""

    def __init__(self, n, batch_size, chunk_size=1, group_chunks=1, shuffle=True, drop_last=False, seed=0):
        self.n, self.bs = n, batch_size
        self.chunk, self.group = max(1, chunk_size), max(1, group_chunks)
        self.shuffle, self.drop_last, self.seed = shuffle, drop_last, seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def _order(self):
        if not self.shuffle:
            return np.arange(self.n)
        rng = np.random.default_rng((self.seed, self.epoch))
        if self.chunk == 1:
            return rng.permutation(self.n)
        starts = rng.permutation(np.arange(0, self.n, self.chunk))
        out = []
        for g in range(0, len(starts), self.group):
            pos = np.concatenate([np.arange(s, min(s + self.chunk, self.n)) for s in starts[g:g + self.group]])
            out.append(rng.permutation(pos))
        return np.concatenate(out)

    def __iter__(self):
        order = self._order()
        end = (self.n // self.bs) * self.bs if self.drop_last else self.n
        for i in range(0, end, self.bs):
            yield order[i:i + self.bs]

    def __len__(self):
        return self.n // self.bs if self.drop_last else -(-self.n // self.bs)


def make_loader(ds, batch_size, shuffle, chunk_size=1, group_chunks=1, seed=0,
                num_workers=2, pin_memory=True, prefetch_factor=PREFETCH_FACTOR):
    sampler = ChunkShuffleBatchSampler(len(ds), batch_size, chunk_size, group_chunks, shuffle, False, seed)
    kw = {}
    if num_workers > 0:
        # fork: worker dùng chung dữ liệu trong RAM, và không cần pickle class định nghĩa trong notebook
        kw = dict(persistent_workers=True, prefetch_factor=prefetch_factor, multiprocessing_context="fork")
    loader = torch.utils.data.DataLoader(ds, sampler=sampler, batch_size=None,
                                         num_workers=num_workers, pin_memory=pin_memory, **kw)
    return loader, sampler


# ---------------- chia tập (giống hệt src/data/splits.py -> cùng file split với máy local) ----------------
def group_ids(labels):
    snr_i = labels[:, COL_SNR].astype(np.int64) - SNR_MIN
    return (snr_i * N_MOD + labels[:, COL_MOD]) * N_DOA + labels[:, COL_DOA]


def _split_units(n_units, rng, test_frac, val_frac):
    perm = rng.permutation(n_units)
    n_test = int(round(n_units * test_frac))
    test, trainval = perm[:n_test], perm[n_test:]
    n_val = int(round(trainval.size * val_frac))
    return trainval[n_val:], trainval[:n_val], test


def make_splits(labels, mode="random", seed=SPLIT_SEED, test_frac=TEST_FRACTION, val_frac=VAL_FRACTION_OF_TRAIN):
    if mode not in SPLIT_MODES:
        raise ValueError(mode)
    rng = np.random.default_rng(seed)
    if mode == "random":
        tr, va, te = _split_units(len(labels), rng, test_frac, val_frac)
    else:
        g = group_ids(labels)
        uniq, inv = np.unique(g, return_inverse=True)
        gtr, gva, gte = _split_units(uniq.size, rng, test_frac, val_frac)
        side = np.empty(uniq.size, dtype=np.int8)
        side[gtr], side[gva], side[gte] = 0, 1, 2
        s = side[inv]
        rows = np.arange(len(labels))
        tr, va, te = rows[s == 0], rows[s == 1], rows[s == 2]
    return {k: np.sort(v).astype(np.int64) for k, v in (("train", tr), ("val", va), ("test", te))}


def split_path(out_dir, mode, seed):
    return os.path.join(out_dir, "splits", f"{mode}_seed{seed}.npz")


def load_or_make_splits(labels, out_dir, mode="random", seed=SPLIT_SEED):
    p = split_path(out_dir, mode, seed)
    if os.path.exists(p):
        z = np.load(p)
        return {k: z[k] for k in ("train", "val", "test")}
    sp = make_splits(labels, mode, seed)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    np.savez(p, **sp)
    return sp


def check_splits(sp, labels, mode):
    tr, va, te = sp["train"], sp["val"], sp["test"]
    assert np.intersect1d(tr, te).size == 0 and np.intersect1d(tr, va).size == 0 and np.intersect1d(va, te).size == 0
    assert tr.size + va.size + te.size == len(labels)
    if mode == "group":
        g = group_ids(labels)
        assert np.intersect1d(g[tr], g[te]).size == 0 and np.intersect1d(g[tr], g[va]).size == 0
    return True

## 5B. (Tuỳ chọn) Convert từ file zip `.mat` gốc
**Chỉ chạy khi `DATA_SOURCE = "mat_zip"`** (không có sẵn `data_processed/`). Cần file zip dataset gốc (~17 GB) trên Drive, tức Drive phải có ≥ 30 GB trống (gói free chỉ 15 GB → không đủ). Các bước: giải nén vào `/content` (~15–30 phút), convert sang memmap fp16 (~30–60 phút với 2 CPU của Colab), rồi chép `data_processed/` về Drive để lần sau dùng `DATA_SOURCE = "processed"`.

In [ ]:
import scipy.io as sio
from concurrent.futures import ProcessPoolExecutor


def scan_dataset(root):
    found = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            i = index_from_filename(f)
            if i is not None:
                found[i] = os.path.join(dirpath, f)
    return found


def load_mat_sample(path):
    """1 file .mat -> (5, 1024, 2) float32 (layout PyTorch)."""
    d = sio.loadmat(path)
    a = d[MAT_VARIABLE]
    if a.shape != RAW_SHAPE:
        raise ValueError(f"{path}: shape {a.shape}, kỳ vọng {RAW_SHAPE}")
    return np.ascontiguousarray(np.transpose(np.asarray(a, dtype=np.float32), PERMUTE_RAW_TO_TORCH))


def _load_many(paths):
    return np.stack([load_mat_sample(p) for p in paths])


def convert_mat_to_memmap(root, out_dir, dtype="float16", batch=CONVERT_BATCH, workers=None):
    """Như src/data/convert.py (bản rút gọn). Ngắt giữa chừng thì chạy lại: tiếp từ progress.json."""
    from tqdm.auto import tqdm
    workers = workers or os.cpu_count()
    found = scan_dataset(root)
    missing = N_TOTAL - len(found)
    if missing:
        raise RuntimeError(f"thiếu {missing} file .mat trong {root}")
    paths = [found[i] for i in range(1, N_TOTAL + 1)]
    os.makedirs(out_dir, exist_ok=True)
    x_path, prog_path = os.path.join(out_dir, f"x_{dtype}.npy"), os.path.join(out_dir, "progress.json")
    shape = (N_TOTAL,) + SAMPLE_SHAPE
    prog = json.load(open(prog_path)) if os.path.exists(prog_path) else {}
    start = prog.get("done", 0) if prog.get("dtype") == dtype and os.path.exists(x_path) else 0
    x = (np.lib.format.open_memmap(x_path, mode="w+", dtype=dtype, shape=shape) if start == 0
         else np.load(x_path, mmap_mode="r+"))
    np.save(os.path.join(out_dir, "labels.npy"), labels_for_all_rows())
    amax, nonfinite = prog.get("amax", 0.0), prog.get("nonfinite", 0)
    sub = max(1, batch // workers)
    with ProcessPoolExecutor(max_workers=workers) as ex, \
            tqdm(total=N_TOTAL, initial=start, unit="file") as bar:
        for s in range(start, N_TOTAL, batch):
            chunk = paths[s:s + batch]
            arr = np.concatenate(list(ex.map(_load_many, [chunk[j:j + sub] for j in range(0, len(chunk), sub)])))
            nonfinite += int((~np.isfinite(arr)).sum())
            amax = max(amax, float(np.abs(arr).max()))
            x[s:s + len(arr)] = arr.astype(dtype)
            x.flush()
            json.dump({"done": s + len(arr), "dtype": dtype, "amax": amax, "nonfinite": nonfinite},
                      open(prog_path, "w"))
            bar.update(len(arr))
    del x
    meta = {"dtype": dtype, "shape": list(shape), "layout": "(row, M, N, IQ)",
            "row_to_index": "row r <-> dataXXXXXXXX.mat with index r+1", "modulations": MODULATIONS,
            "label_columns": list(LABEL_COLUMNS), "source_root": os.path.abspath(root),
            "stats": {"amax": amax, "nonfinite": nonfinite}, "size_bytes": os.path.getsize(x_path),
            "complete": True}
    json.dump(meta, open(os.path.join(out_dir, "meta.json"), "w"), indent=2)
    return meta


if DATA_SOURCE == "mat_zip":
    MAT_ROOT = "/content/MoDANet-dataset"
    if not os.path.exists(os.path.join(LOCAL_DATA, "meta.json")):
        if not os.path.isdir(MAT_ROOT):
            print("Giải nén", MAT_ZIP, "->", MAT_ROOT, "(15–30 phút) ...")
            t = time.time()
            os.makedirs(MAT_ROOT, exist_ok=True)
            !unzip -q -o "{MAT_ZIP}" -d "{MAT_ROOT}"
            print(f"  xong ({time.time() - t:.0f}s)")
        meta = convert_mat_to_memmap(MAT_ROOT, LOCAL_DATA)
        print("Convert xong:", meta["stats"], f"{meta['size_bytes'] / 1e9:.2f} GB")
        load_or_make_splits(np.load(os.path.join(LOCAL_DATA, "labels.npy")), LOCAL_DATA, "random", SPLIT_SEED)
    # lưu về Drive để lần sau dùng DATA_SOURCE = "processed" (bỏ qua nếu Drive không đủ chỗ)
    for f in ["meta.json", "labels.npy", "splits/random_seed42.npz", "x_float16.npy"]:
        copy_with_progress(os.path.join(LOCAL_DATA, f), os.path.join(SRC_PROCESSED, f))
    print("Đã lưu data_processed/ lên Drive. Có thể xoá", MAT_ROOT, "cho nhẹ đĩa: !rm -rf", MAT_ROOT)
else:
    print("DATA_SOURCE =", DATA_SOURCE, "-> bỏ qua mục 5B")

## 6. Code: tiện ích (thiết bị, seed, checkpoint, metric)

In [ ]:
import random
import shutil as _shutil


def _kernel_test(device):
    """Chạy kernel thật (conv + pool + backward) để bắt lỗi 'no kernel image is available'."""
    try:
        x = torch.randn(2, 5, 64, 2, device=device)
        conv = torch.nn.Conv2d(5, 8, (7, 1), padding=(3, 0)).to(device)
        torch.nn.functional.max_pool2d(conv(x), (3, 1), (2, 1), (1, 0)).sum().backward()
        torch.cuda.synchronize(device)
        return "OK"
    except RuntimeError as e:
        return f"FAIL: {e}"


def select_device(pref="auto", verbose=True):
    if pref == "cpu":
        return torch.device("cpu")
    if torch.cuda.is_available():
        kt = _kernel_test(torch.device("cuda"))
        if kt == "OK":
            if verbose:
                p = torch.cuda.get_device_properties(0)
                print(f"GPU: {p.name} cap={torch.cuda.get_device_capability(0)} VRAM={p.total_memory / 2**30:.1f} GB "
                      f"| torch {torch.__version__} cu{torch.version.cuda}")
            return torch.device(pref if pref.startswith("cuda:") else "cuda")
        print("⚠️  GPU không chạy được kernel:", kt)
    if pref.startswith("cuda"):
        raise RuntimeError("Yêu cầu CUDA nhưng không có GPU dùng được (Runtime -> Change runtime type -> T4 GPU)")
    if verbose:
        print("⚠️  Không có GPU -> CPU (train đầy đủ sẽ RẤT chậm). Colab: Runtime -> Change runtime type -> T4 GPU")
    return torch.device("cpu")


def auto_micro_batch(device, effective=EFFECTIVE_BATCH):
    """Micro-batch theo VRAM (instruction 9.3 + mở rộng). Batch hiệu dụng luôn = effective."""
    device = torch.device(device)
    if device.type != "cuda":
        return effective
    gb = torch.cuda.get_device_properties(device).total_memory / 2**30
    for min_gb, micro in AUTO_BATCH_BY_VRAM_GB:
        if gb >= min_gb:
            return min(micro, effective)
    return min(AUTO_BATCH_BY_VRAM_GB[-1][1], effective)


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:
        return os.cpu_count() or 1


def resolve_num_workers(requested, batch_bytes, verbose=True):
    """'auto' -> min(4, số CPU - 1), hạ nếu /dev/shm nhỏ. Không có fork -> 0."""
    import multiprocessing
    if "fork" not in multiprocessing.get_all_start_methods():
        return 0
    if requested != "auto":
        return int(requested)
    n = max(0, min(MAX_WORKERS, available_cpus() - 1))
    try:
        free = _shutil.disk_usage("/dev/shm").free
    except OSError:
        free = None
    while free is not None and n > 0 and n * PREFETCH_FACTOR * batch_bytes * 2 > free:
        n -= 1
    return n


def mem_available_bytes():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) * 1024
    except OSError:
        pass
    return None


def seed_everything(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic


def get_rng_state():
    # trạng thái numpy lưu dạng list (không phải ndarray) -> checkpoint nạp được với cả NumPy 1.x lẫn 2.x
    name, keys, pos, has_gauss, gauss = np.random.get_state()
    st = {"python": random.getstate(),
          "numpy": (name, keys.tolist(), int(pos), int(has_gauss), float(gauss)),
          "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st


def set_rng_state(st):
    random.setstate(st["python"])
    name, keys, pos, has_gauss, gauss = st["numpy"]
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), pos, has_gauss, gauss))
    torch.set_rng_state(st["torch"])
    if "cuda" in st and torch.cuda.is_available():
        for i in range(min(len(st["cuda"]), torch.cuda.device_count())):
            torch.cuda.set_rng_state(st["cuda"][i], i)


def save_checkpoint(path, **state):
    """Ghi nguyên tử (tmp -> rename): ngắt giữa chừng không làm hỏng file cũ."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, map_location="cpu"):
    return torch.load(path, map_location=map_location, weights_only=False)


def build_model_from_checkpoint(ckpt):
    model = MoDANet(**ckpt["config_snapshot"]["model"])
    model.load_state_dict(ckpt["model_state"])
    return model


def rmse_deg(pred_doa_idx, true_doa_idx):
    a = idx_to_angle(np.asarray(pred_doa_idx)) - idx_to_angle(np.asarray(true_doa_idx))
    return float(np.sqrt(np.mean(a.astype(np.float64) ** 2))) if a.size else float("nan")


def fmt_time(s):
    s = int(s)
    h, r = divmod(s, 3600)
    m, s = divmod(r, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"


DEVICE = select_device("auto")

## 7. Code: vòng train (scripts/train.py)
`make_args(...)` có đúng các tham số của `scripts/train.py` nên checkpoint dùng chung được với máy local. Mỗi epoch: ghi 1 dòng `log.csv` (flush + fsync), lưu `last.pt` và `best.pt` (theo val_loss) lên Drive.

In [ ]:
import csv
import math
from types import SimpleNamespace

CSV_FIELDS = ["epoch", "lr", "train_loss", "train_loss_amc", "train_loss_doa",
              "val_loss", "val_loss_amc", "val_loss_doa", "val_acc_amc", "val_acc_doa",
              "val_rmse_deg", "gap", "epoch_time"]
# được phép đổi khi resume (còn lại lấy từ checkpoint để tái lập đúng lần chạy) — giống scripts/train.py
RUNTIME_ARGS = ("max_hours", "num_workers", "device", "data_dir", "epochs", "patience",
                "no_early_stop", "log_every", "batch_size", "accum_steps", "in_memory")


def make_args(**kw):
    """Tương đương parse_args() của scripts/train.py (cùng tên tham số)."""
    a = dict(preset=DEFAULT_PRESET, filters=None, filter_size=None, variant=None, batch_size=None,
             accum_steps=None, lr=None, epochs=None, out=None, resume=None, smoke=False, subset=None,
             split_mode="random", seed=SEED, patience=PATIENCE, no_early_stop=False, max_hours=None,
             data_dir=LOCAL_DATA, num_workers="auto", device="auto", in_memory=False,
             shuffle="chunk", deterministic=False, log_every=200)
    bad = set(kw) - set(a)
    if bad:
        raise TypeError(f"tham số không có: {sorted(bad)}")
    a.update(kw)
    if a["smoke"]:
        a["subset"] = a["subset"] or SMOKE_SUBSET
        a["epochs"] = a["epochs"] or SMOKE_EPOCHS
        a["out"] = a["out"] or os.path.join(OUT_ROOT, "smoke")
    a["epochs"] = a["epochs"] or TRAIN_EPOCHS
    a["out"] = a["out"] or os.path.join(OUT_ROOT, "run01")
    return SimpleNamespace(**a)


def resolve_batch(args, device, saved_effective=None):
    bs = args.batch_size if args.batch_size is not None else "auto"
    if saved_effective:
        micro = auto_micro_batch(device, saved_effective) if bs == "auto" else bs
        if saved_effective % micro:
            raise ValueError(f"batch_size {micro} không chia hết batch hiệu dụng {saved_effective} của checkpoint")
        return micro, saved_effective // micro
    if bs == "auto":
        micro = auto_micro_batch(device)
        return micro, args.accum_steps or max(1, EFFECTIVE_BATCH // micro)
    return bs, args.accum_steps or 1


def model_hparams(args):
    p = dict(PRESETS[args.preset])
    return dict(K=args.filters or p["K"], k=args.filter_size or p["k"], variant=args.variant or p["variant"])


def acc_dtype(device):
    return torch.float32 if device.type == "mps" else torch.float64


@torch.no_grad()
def evaluate(model, loader, crit, device):
    model.eval()
    n = 0
    s_loss = torch.zeros(3, dtype=acc_dtype(device), device=device)
    hits = torch.zeros(2, dtype=torch.long, device=device)
    pd, td = [], []
    for x, ym, yd, _ in loader:
        x, ym, yd = x.to(device, non_blocking=True), ym.to(device), yd.to(device)
        la, ld = model(x)
        loss, l_amc, l_doa = crit(la, ld, ym, yd)
        b = x.shape[0]
        n += b
        s_loss += torch.stack((loss, l_amc, l_doa)).to(s_loss.dtype) * b
        pdoa = ld.argmax(1)
        hits += torch.stack(((la.argmax(1) == ym).sum(), (pdoa == yd).sum()))
        pd.append(pdoa)
        td.append(yd)
    l_sum, l_amc, l_doa = (s_loss / n).tolist()
    c_amc, c_doa = hits.tolist()
    return dict(val_loss=l_sum, val_loss_amc=l_amc, val_loss_doa=l_doa, val_acc_amc=c_amc / n,
                val_acc_doa=c_doa / n,
                val_rmse_deg=rmse_deg(torch.cat(pd).cpu().numpy(), torch.cat(td).cpu().numpy()))


_X_CACHE = {}   # giữ dữ liệu đã nạp RAM giữa các lần gọi train() (smoke -> full) để không nạp lại 9.2 GB


def get_data(data_dir, in_memory):
    key = (os.path.abspath(data_dir), bool(in_memory))
    if key not in _X_CACHE:
        _X_CACHE.clear()
        x, labels, meta = open_processed(data_dir)
        if in_memory:
            t = time.time()
            x = np.array(x)
            print(f"Đã nạp {x.nbytes / 2**30:.1f} GB vào RAM trong {time.time() - t:.0f}s")
        _X_CACHE[key] = (x, labels, meta)
    return _X_CACHE[key]


def train(args):
    """Trả về đường dẫn thư mục run. Ngắt (■ Interrupt) lúc nào cũng được: chạy lại để resume."""
    ckpt = None
    if args.resume:
        ckpt = load_checkpoint(args.resume)
        cli = vars(args)
        for k, v in ckpt["config_snapshot"]["args"].items():
            if k not in RUNTIME_ARGS and k != "resume":
                cli[k] = v
        args.out = os.path.dirname(os.path.abspath(args.resume))
        print(f"Resume từ {args.resume} (đã xong {ckpt['epoch']} epoch)")
        if ckpt["epoch"] >= args.epochs or (not args.no_early_stop and ckpt["epochs_no_improve"] >= args.patience):
            print(f"Run này đã kết thúc (epoch {ckpt['epoch']}, {ckpt['epochs_no_improve']} epoch không cải thiện) "
                  f"-> không train thêm. Muốn train lại từ đầu: đổi RUN_NAME.")
            return args.out
    os.makedirs(args.out, exist_ok=True)

    seed_everything(args.seed, args.deterministic)
    device = select_device(args.device, verbose=False)
    saved_eff = ckpt["config_snapshot"].get("effective_batch") if ckpt else None
    args.batch_size, args.accum_steps = resolve_batch(args, device, saved_eff)

    # ---- data
    x_mm, labels, meta = get_data(args.data_dir, args.in_memory)
    batch_bytes = args.batch_size * int(np.prod(SAMPLE_SHAPE)) * 4
    args.num_workers = resolve_num_workers(args.num_workers, batch_bytes)
    sp = load_or_make_splits(labels, args.data_dir, args.split_mode, SPLIT_SEED)
    check_splits(sp, labels, args.split_mode)
    tr_rows, va_rows = sp["train"], sp["val"]
    if args.subset:
        rng = np.random.default_rng(args.seed)
        n_val = max(args.batch_size, int(round(args.subset * VAL_FRACTION_OF_TRAIN / (1 - VAL_FRACTION_OF_TRAIN))))
        tr_rows = np.sort(rng.choice(tr_rows, min(args.subset, tr_rows.size), replace=False))
        va_rows = np.sort(rng.choice(va_rows, min(n_val, va_rows.size), replace=False))
    tr_ds, va_ds = ModaNetDataset(x_mm, labels, tr_rows), ModaNetDataset(x_mm, labels, va_rows)
    chunk, group = (CHUNK_SIZE, SHUFFLE_GROUP_CHUNKS) if args.shuffle == "chunk" else (1, 1)
    pin = PIN_MEMORY and device.type == "cuda"
    tr_loader, tr_sampler = make_loader(tr_ds, args.batch_size, True, chunk, group, args.seed, args.num_workers, pin)
    va_loader, _ = make_loader(va_ds, args.batch_size, False, num_workers=args.num_workers, pin_memory=pin)

    # ---- model
    model = MoDANet(**model_hparams(args)).to(device)
    crit = MultiTaskLoss(LOSS_WEIGHT_AMC, LOSS_WEIGHT_DOA)
    eff_bs = args.batch_size * args.accum_steps
    lr = args.lr if args.lr is not None else LR * eff_bs / EFFECTIVE_BATCH
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM)
    sched = torch.optim.lr_scheduler.StepLR(opt, LR_DROP_EPOCH, LR_DROP_FACTOR)

    start_epoch, best_val, bad_epochs, history = 0, math.inf, 0, []
    if ckpt:
        model.load_state_dict(ckpt["model_state"])
        opt.load_state_dict(ckpt["optimizer_state"])
        sched.load_state_dict(ckpt["scheduler_state"])
        set_rng_state(ckpt["rng_state"])
        start_epoch, best_val = ckpt["epoch"], ckpt["best_val_loss"]
        bad_epochs, history = ckpt["epochs_no_improve"], ckpt.get("history", [])

    snapshot = {"model": model.hparams, "preset": args.preset, "args": vars(args).copy(),
                "modulations": MODULATIONS, "data_dtype": meta["dtype"],
                "split_file": split_path(args.data_dir, args.split_mode, SPLIT_SEED),
                "effective_batch": eff_bs, "lr0": lr}

    print(f"Model: preset={args.preset} {model.hparams} | {count_params(model):,} tham số")
    print(f"Dữ liệu: train {len(tr_ds):,} | val {len(va_ds):,} | split={args.split_mode} | shuffle={args.shuffle} "
          f"| {meta['dtype']} | {'RAM' if args.in_memory else 'memmap'} | workers {args.num_workers}")
    print(f"Tối ưu: SGD lr={lr:g} momentum={MOMENTUM} | batch {args.batch_size}×{args.accum_steps}={eff_bs} "
          f"| lr×{LR_DROP_FACTOR} tại epoch {LR_DROP_EPOCH} | epochs {args.epochs} "
          f"| early stop {'tắt' if args.no_early_stop else f'patience {args.patience}'}")
    print(f"Ghi vào: {args.out}")

    csv_path = os.path.join(args.out, "log.csv")
    new_csv = not os.path.exists(csv_path) or start_epoch == 0
    csv_f = open(csv_path, "w" if new_csv else "a", newline="")
    writer = csv.DictWriter(csv_f, fieldnames=CSV_FIELDS)
    if new_csv:
        writer.writeheader()
        csv_f.flush()

    print(f"{'epoch':>5}  {'lr':>7}  {'train_loss':>10}  {'val_loss':>8}  {'gap':>7}  "
          f"{'val_acc_amc':>11}  {'val_acc_doa':>11}  {'rmse°':>6}  {'time':>7}  {'ETA':>7}")
    t_run = time.time()
    n_iter = len(tr_loader)
    epoch_times = []

    def ckpt_state(epoch):
        return dict(epoch=epoch, model_state=model.state_dict(), optimizer_state=opt.state_dict(),
                    scheduler_state=sched.state_dict(), rng_state=get_rng_state(), config_snapshot=snapshot,
                    best_val_loss=best_val, epochs_no_improve=bad_epochs, history=history)

    stop_reason = "đủ số epoch"
    try:
        for epoch in range(start_epoch, args.epochs):
            t0 = time.time()
            model.train()
            tr_sampler.set_epoch(epoch)
            n = 0
            s = torch.zeros(3, dtype=acc_dtype(device), device=device)
            opt.zero_grad(set_to_none=True)
            for it, (x, ym, yd, _) in enumerate(tr_loader):
                x = x.to(device, non_blocking=True)
                ym, yd = ym.to(device, non_blocking=True), yd.to(device, non_blocking=True)
                la, ld = model(x)
                loss, l_amc, l_doa = crit(la, ld, ym, yd)
                (loss / args.accum_steps).backward()
                if (it + 1) % args.accum_steps == 0 or it + 1 == n_iter:
                    opt.step()
                    opt.zero_grad(set_to_none=True)
                b = x.shape[0]
                n += b
                s += torch.stack((loss.detach(), l_amc.detach(), l_doa.detach())).to(s.dtype) * b
                if args.log_every and (it + 1) % args.log_every == 0:
                    el = time.time() - t0
                    print(f"   [{epoch + 1}] it {it + 1}/{n_iter}  loss {s[0].item() / n:.4f}  "
                          f"{el / (it + 1) * 1000:.0f} ms/it  còn ~{fmt_time(el / (it + 1) * (n_iter - it - 1))}",
                          flush=True)
            s_loss, s_amc, s_doa = (s / n).tolist()
            cur_lr = opt.param_groups[0]["lr"]
            sched.step()
            ev = evaluate(model, va_loader, crit, device)
            dt = time.time() - t0
            epoch_times.append(dt)
            row = dict(epoch=epoch + 1, lr=cur_lr, train_loss=s_loss, train_loss_amc=s_amc, train_loss_doa=s_doa,
                       gap=ev["val_loss"] - s_loss, epoch_time=dt, **ev)
            history.append(row)
            writer.writerow({k: (f"{v:.6g}" if isinstance(v, float) else v) for k, v in row.items()})
            csv_f.flush()
            os.fsync(csv_f.fileno())

            improved = ev["val_loss"] < best_val
            if improved:
                best_val, bad_epochs = ev["val_loss"], 0
            else:
                bad_epochs += 1
            eta = np.mean(epoch_times) * (args.epochs - epoch - 1)
            print(f"{epoch + 1:>5}  {cur_lr:7.4f}  {row['train_loss']:10.4f}  {ev['val_loss']:8.4f}  "
                  f"{row['gap']:+7.3f}  {ev['val_acc_amc'] * 100:10.1f}%  {ev['val_acc_doa'] * 100:10.1f}%  "
                  f"{ev['val_rmse_deg']:6.2f}  {fmt_time(dt):>7}  {fmt_time(eta):>7}"
                  f"{'  *best' if improved else ''}", flush=True)

            save_checkpoint(os.path.join(args.out, "last.pt"), **ckpt_state(epoch + 1))
            if improved:
                save_checkpoint(os.path.join(args.out, "best.pt"), **ckpt_state(epoch + 1))

            W = OVERFIT_WARN_EPOCHS
            if len(history) > W:
                h = history[-(W + 1):]
                if all(h[i + 1]["val_loss"] > h[i]["val_loss"] for i in range(W)) and \
                   all(h[i + 1]["train_loss"] < h[i]["train_loss"] for i in range(W)):
                    print(f"⚠️  val_loss tăng {W} epoch liên tục trong khi train_loss giảm -> dấu hiệu overfitting")

            if not args.no_early_stop and bad_epochs >= args.patience:
                stop_reason = f"early stopping (val_loss không cải thiện {args.patience} epoch)"
                break
            if args.max_hours is not None and time.time() - t_run + np.mean(epoch_times) > args.max_hours * 3600:
                stop_reason = f"hết ngân sách MAX_HOURS = {args.max_hours} (chạy lại cell để train tiếp)"
                break
    except KeyboardInterrupt:
        print("\nĐã ngắt. Các epoch đã xong vẫn ở last.pt — chạy lại cell này để train tiếp.")
        raise   # dừng luôn "Run all", không chạy tiếp phần đánh giá
    finally:
        csv_f.close()
    print(f"Dừng: {stop_reason}. best val_loss = {best_val:.4f} | tổng {fmt_time(time.time() - t_run)}")
    if epoch_times:
        print(f"Thời gian/epoch đo được: {fmt_time(np.mean(epoch_times))} ({len(tr_ds):,} mẫu train, {device.type})")
    return args.out


def resolve_in_memory(setting, data_dir, headroom_gb=3.0):
    """'auto': chỉ nạp RAM khi còn dư >= headroom_gb sau khi nạp (Colab hết RAM là crash cả phiên)."""
    if setting != "auto":
        return bool(setting)
    need = os.path.getsize(os.path.join(data_dir, "x_float16.npy"))
    avail = mem_available_bytes()
    ok = avail is not None and avail > need + headroom_gb * 2**30
    print(f"IN_MEMORY auto: RAM trống {0 if avail is None else avail / 2**30:.1f} GB, dữ liệu {need / 2**30:.1f} GB "
          f"-> {'nạp vào RAM' if ok else 'đọc memmap từ đĩa'}")
    return ok


def run_training(run_name, **overrides):
    """Tự resume nếu outputs/<run_name>/last.pt đã có (kể cả last.pt upload từ máy local)."""
    out = os.path.join(OUT_ROOT, run_name)
    last = os.path.join(out, "last.pt")
    args = make_args(out=out, resume=last if os.path.exists(last) else None, **overrides)
    return train(args)

## 8. Chạy thử nhanh (smoke test)
1 epoch trên 20,000 mẫu. Dòng cuối in **thời gian/epoch**; nhân với ~16 (324,086 / 20,000) rồi × 60 để ước lượng thời gian train đầy đủ. Epoch đầu có thể chậm hơn vì đọc đĩa lần đầu. Kết quả smoke ghi vào `outputs/smoke/` và **luôn chạy lại từ đầu**.

In [ ]:
IN_MEM = resolve_in_memory(IN_MEMORY, LOCAL_DATA)
# BATCH_SIZE là số -> accum = 128 // BATCH_SIZE để batch hiệu dụng vẫn là 128
ACCUM = None if BATCH_SIZE == "auto" else max(1, EFFECTIVE_BATCH // int(BATCH_SIZE))
if RUN_SMOKE:
    _smoke_dir = os.path.join(OUT_ROOT, "smoke")
    if os.path.isdir(_smoke_dir):
        _shutil.rmtree(_smoke_dir)
    train(make_args(smoke=True, subset=20000, epochs=1, out=_smoke_dir, batch_size=BATCH_SIZE, accum_steps=ACCUM,
                    num_workers=NUM_WORKERS, in_memory=IN_MEM, shuffle=SHUFFLE, log_every=50))

## 9. Train đầy đủ
- Có thể đóng tab trình duyệt? **Không** với Colab free: phiên bị ngắt sau ~90 phút không tương tác, và tối đa ~12 giờ. Giữ tab mở. Colab Pro có tuỳ chọn chạy nền.
- Mỗi epoch được lưu lên Drive. Bị ngắt thì: **Runtime → Run all** (hoặc chạy lại mục 1 → 9). Notebook sẽ chép lại dữ liệu nếu máy mới, rồi **tự resume** từ `last.pt`.
- Muốn dừng sớm: bấm ■ (Interrupt). Epoch đang chạy dở bị bỏ, các epoch đã xong vẫn còn.
- Muốn train tiếp một run bắt đầu ở máy local: upload `outputs/run01/last.pt` (và `log.csv`) vào `MyDrive/modanet/outputs/run01/` rồi chạy cell này. Batch hiệu dụng 128 và lịch lr được giữ nguyên.

In [ ]:
if RUN_FULL:
    RUN_DIR = run_training(RUN_NAME, preset=PRESET, split_mode=SPLIT_MODE, epochs=EPOCHS, batch_size=BATCH_SIZE,
                           accum_steps=ACCUM, no_early_stop=NO_EARLY_STOP, max_hours=MAX_HOURS, num_workers=NUM_WORKERS,
                           in_memory=IN_MEM, shuffle=SHUFFLE, log_every=LOG_EVERY)
else:
    RUN_DIR = os.path.join(OUT_ROOT, RUN_NAME)

## 10. Đường cong train (từ `log.csv`)

In [ ]:
import matplotlib.pyplot as plt


def plot_log(run_dir):
    p = os.path.join(run_dir, "log.csv")
    if not os.path.exists(p):
        print("Chưa có", p)
        return
    rows = list(csv.DictReader(open(p)))
    if not rows:
        print("log.csv rỗng")
        return
    col = lambda k: np.array([float(r[k]) for r in rows])
    ep = col("epoch")
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
    ax[0].plot(ep, col("train_loss"), label="train"); ax[0].plot(ep, col("val_loss"), label="val")
    ax[0].set_title("loss (AMC + DOA)"); ax[0].legend()
    ax[1].plot(ep, col("val_acc_amc") * 100); ax[1].set_title("val AMC accuracy (%)")
    ax[2].plot(ep, col("val_acc_doa") * 100); ax[2].set_title("val DOA accuracy (%)")
    ax[3].plot(ep, col("val_rmse_deg")); ax[3].set_title("val DOA RMSE (°)")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=.3)
    plt.tight_layout()
    fig.savefig(os.path.join(run_dir, "train_curves.png"), dpi=120)
    plt.show()
    best = min(rows, key=lambda r: float(r["val_loss"]))
    print(f"{len(rows)} epoch | best epoch {best['epoch']}: val_loss {float(best['val_loss']):.4f}, "
          f"AMC {float(best['val_acc_amc']) * 100:.2f}%, DOA {float(best['val_acc_doa']) * 100:.2f}%, "
          f"RMSE {float(best['val_rmse_deg']):.3f}°")


plot_log(RUN_DIR)

## 11. Đánh giá `best.pt` trên tập test
Accuracy AMC, accuracy DOA (đúng góc 1°) và RMSE DOA theo từng mức SNR, cộng confusion matrix AMC. Tập test là 20% dữ liệu, không dùng khi train hay chọn checkpoint. Kết quả lưu vào `outputs/<RUN_NAME>/test_*.{json,csv,png}`.

In [ ]:
@torch.no_grad()
def predict_rows(model, x, labels, rows, device, batch=512, num_workers=2):
    ds = ModaNetDataset(x, labels, rows)
    loader, _ = make_loader(ds, batch, False, num_workers=resolve_num_workers(num_workers, 0),
                            pin_memory=device.type == "cuda")
    model.eval()
    pm, pd, tm, td, snr = [], [], [], [], []
    for xb, ym, yd, s in loader:
        la, ld = model(xb.to(device, non_blocking=True))
        pm.append(la.argmax(1).cpu()); pd.append(ld.argmax(1).cpu())
        tm.append(ym); td.append(yd); snr.append(s)
    cat = lambda v: torch.cat(v).numpy()
    return cat(pm), cat(pd), cat(tm), cat(td), cat(snr)


def evaluate_test(run_dir, ckpt_name="best.pt", split="test"):
    ckpt = load_checkpoint(os.path.join(run_dir, ckpt_name))
    model = build_model_from_checkpoint(ckpt).to(DEVICE)
    x, labels, _ = get_data(LOCAL_DATA, IN_MEM)
    mode = ckpt["config_snapshot"]["args"].get("split_mode", "random")
    rows = load_or_make_splits(labels, LOCAL_DATA, mode, SPLIT_SEED)[split]
    t = time.time()
    pm, pd, tm, td, snr = predict_rows(model, x, labels, rows, DEVICE)
    print(f"{ckpt_name} (epoch {ckpt['epoch']}) trên {split}: {len(rows):,} mẫu, {time.time() - t:.0f}s")
    per_snr = []
    for s in SNR_LEVELS:
        k = snr == s
        per_snr.append(dict(snr_db=s, n=int(k.sum()), acc_amc=float((pm[k] == tm[k]).mean()),
                            acc_doa=float((pd[k] == td[k]).mean()), rmse_deg=rmse_deg(pd[k], td[k])))
    overall = dict(n=len(rows), acc_amc=float((pm == tm).mean()), acc_doa=float((pd == td).mean()),
                   rmse_deg=rmse_deg(pd, td), epoch=ckpt["epoch"], ckpt=ckpt_name, split=split, split_mode=mode)
    cm = np.zeros((N_MOD, N_MOD), dtype=np.int64)
    np.add.at(cm, (tm, pm), 1)
    per_class = {MODULATIONS[i]: float(cm[i, i] / max(1, cm[i].sum())) for i in range(N_MOD)}

    print(f"TỔNG: AMC {overall['acc_amc'] * 100:.2f}% | DOA {overall['acc_doa'] * 100:.2f}% | "
          f"RMSE {overall['rmse_deg']:.3f}°")
    print(f"{'SNR':>5} {'AMC%':>7} {'DOA%':>7} {'RMSE°':>7}")
    for r in per_snr:
        print(f"{r['snr_db']:>+5d} {r['acc_amc'] * 100:7.2f} {r['acc_doa'] * 100:7.2f} {r['rmse_deg']:7.3f}")

    json.dump(dict(overall=overall, per_snr=per_snr, per_class_acc_amc=per_class,
                   confusion_matrix=cm.tolist(), modulations=MODULATIONS),
              open(os.path.join(run_dir, f"{split}_results.json"), "w"), indent=2)
    with open(os.path.join(run_dir, f"{split}_per_snr.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(per_snr[0]))
        w.writeheader(); w.writerows(per_snr)

    S = [r["snr_db"] for r in per_snr]
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    ax[0].plot(S, [r["acc_amc"] * 100 for r in per_snr], "o-"); ax[0].set_title("AMC accuracy (%) vs SNR")
    ax[1].plot(S, [r["acc_doa"] * 100 for r in per_snr], "o-"); ax[1].set_title("DOA accuracy (%) vs SNR")
    ax[2].semilogy(S, [max(r["rmse_deg"], 1e-3) for r in per_snr], "o-"); ax[2].set_title("DOA RMSE (°) vs SNR")
    for a in ax:
        a.set_xlabel("SNR (dB)"); a.grid(alpha=.3, which="both")
    plt.tight_layout(); fig.savefig(os.path.join(run_dir, f"{split}_vs_snr.png"), dpi=120); plt.show()

    cmn = cm / np.maximum(1, cm.sum(1, keepdims=True))
    fig, a = plt.subplots(figsize=(8, 7))
    im = a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(N_MOD)); a.set_xticklabels(MODULATIONS, rotation=60, ha="right")
    a.set_yticks(range(N_MOD)); a.set_yticklabels(MODULATIONS)
    a.set_xlabel("dự đoán"); a.set_ylabel("thật"); a.set_title(f"Confusion matrix AMC ({split}, mọi SNR)")
    for i in range(N_MOD):
        for j in range(N_MOD):
            if cmn[i, j] >= 0.01:
                a.text(j, i, f"{cmn[i, j] * 100:.0f}", ha="center", va="center", fontsize=7,
                       color="white" if cmn[i, j] > 0.5 else "black")
    fig.colorbar(im, fraction=0.046); plt.tight_layout()
    fig.savefig(os.path.join(run_dir, f"{split}_confusion_amc.png"), dpi=120); plt.show()
    return overall, per_snr


if RUN_EVAL and os.path.exists(os.path.join(RUN_DIR, "best.pt")):
    TEST_OVERALL, TEST_PER_SNR = evaluate_test(RUN_DIR)
elif RUN_EVAL:
    print("Chưa có", os.path.join(RUN_DIR, "best.pt"))

## 12. Suy luận (scripts/inference.py)
- `infer_split(...)`: lấy ngẫu nhiên vài mẫu test, in dự đoán kèm nhãn thật.
- `infer_mat(path)`: dự đoán một file `.mat` (upload vào `/content` hoặc Drive).
- `benchmark(...)`: thời gian suy luận ms/frame (bài báo Fig. 6b ~1.9 ms/frame).

In [ ]:
@torch.no_grad()
def predict_proba(model, x, device, batch=256):
    pa, pd = [], []
    for i in range(0, len(x), batch):
        la, ld = model(torch.from_numpy(np.asarray(x[i:i + batch], dtype=np.float32)).to(device))
        pa.append(torch.softmax(la, 1).cpu()); pd.append(torch.softmax(ld, 1).cpu())
    return torch.cat(pa).numpy(), torch.cat(pd).numpy()


def show_predictions(pa, pd, names, truth=None, topk=3):
    for i, name in enumerate(names):
        m, d = int(pa[i].argmax()), int(pd[i].argmax())
        alt = ", ".join(f"{MODULATIONS[j]} {pa[i, j] * 100:.1f}%" for j in np.argsort(-pa[i])[1:topk])
        line = f"{name:18s} mod: {MODULATIONS[m]:7s} ({pa[i, m] * 100:5.1f}%)  DOA: {idx_to_angle(d):+4d}° ({pd[i, d] * 100:5.1f}%)"
        if truth is not None:
            tm, td = truth[i]
            line += f"  | thật: {MODULATIONS[tm]:7s} {idx_to_angle(td):+4d}° {'✓' if m == tm else '✗'}{'✓' if d == td else '✗'}"
        print(line + f"   [khác: {alt}]")


def load_model(run_dir=None, ckpt_name="best.pt"):
    ckpt = load_checkpoint(os.path.join(run_dir or RUN_DIR, ckpt_name))
    return build_model_from_checkpoint(ckpt).to(DEVICE).eval(), ckpt


def infer_split(num=16, split="test", run_dir=None, seed=0):
    model, ckpt = load_model(run_dir)
    x, labels, _ = get_data(LOCAL_DATA, IN_MEM)
    mode = ckpt["config_snapshot"]["args"].get("split_mode", "random")
    rows = load_or_make_splits(labels, LOCAL_DATA, mode, SPLIT_SEED)[split]
    rows = np.sort(np.random.default_rng(seed).choice(rows, min(num, rows.size), replace=False))
    pa, pd = predict_proba(model, x[rows], DEVICE)
    show_predictions(pa, pd, [f"data{r + 1:08d}.mat" for r in rows],
                     [(int(labels[r, COL_MOD]), int(labels[r, COL_DOA])) for r in rows])


def infer_mat(path, run_dir=None):
    model, _ = load_model(run_dir)
    x = load_mat_sample(path)[None]
    pa, pd = predict_proba(model, x, DEVICE)
    i = index_from_filename(path)
    truth = [decode_index(i)[1:3]] if i is not None else None
    show_predictions(pa, pd, [os.path.basename(path)], truth)


@torch.no_grad()
def benchmark(n_frames=2000, batch=1, run_dir=None):
    model, _ = load_model(run_dir)
    x = torch.randn((batch,) + SAMPLE_SHAPE, device=DEVICE)
    for _ in range(5):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    reps = max(1, n_frames // batch)
    t = time.perf_counter()
    for _ in range(reps):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t) / (reps * batch) * 1000
    print(f"Suy luận: {ms:.3f} ms/frame (batch {batch}, {DEVICE.type}) — bài báo Fig.6b ~1.9 ms/frame")
    return ms


if os.path.exists(os.path.join(RUN_DIR, "best.pt")):
    infer_split(16)
    benchmark(2000, batch=1)
# infer_mat("/content/drive/MyDrive/modanet/data00199051.mat")   # ví dụ: dự đoán 1 file .mat

## 13. Lấy kết quả về máy
Mọi thứ đã nằm trên Drive: `MyDrive/modanet/outputs/<RUN_NAME>/` gồm `log.csv`, `last.pt`, `best.pt`, `train_curves.png`, `test_results.json`, `test_per_snr.csv`, `test_vs_snr.png`, `test_confusion_amc.png`.

Tải về thư mục `modanet/outputs/run01/` ở máy local rồi dùng như checkpoint train tại máy:
```bash
python scripts/inference.py --ckpt outputs/run01/best.pt --from-split test --num 32
python scripts/train.py --resume outputs/run01/last.pt      # train tiếp trên máy local (nếu chưa xong)
```
Cell dưới nén thư mục run (bỏ `last.pt`) để tải thẳng qua trình duyệt nếu cần.

In [ ]:
if IN_COLAB and os.path.isdir(RUN_DIR):
    _zip = f"/content/{RUN_NAME}_results.zip"
    !cd "{RUN_DIR}" && zip -q -r "{_zip}" . -x "last.pt" -x "*.tmp"
    from google.colab import files
    files.download(_zip)